# Model development

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data.load_data import load_raw_data, split_features_target
from src.utils.config import CV_FOLDS, RANDOM_STATE, TEST_SIZE

## Train/test split

In [3]:
df = load_raw_data()
X, y = split_features_target(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3)

((32000, 13), (8000, 13), np.float64(0.072), np.float64(0.072))

The split is stratified on `y`, so both sets keep the 7.2% subscription rate. Cross-validation runs on the training set only. The test set is scored once, after the final model is chosen.

## Cross-validation setup

In [4]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": make_scorer(f1_score, zero_division=0),
}

## Baseline

In [5]:
baseline = DummyClassifier(strategy="most_frequent")
baseline_scores = cross_validate(baseline, X_train, y_train, cv=cv, scoring=scoring)

pd.DataFrame(baseline_scores).filter(like="test_").mean().round(3)

test_accuracy     0.928
test_precision    0.000
test_recall       0.000
test_f1           0.000
dtype: float64

Always predicting "no" scores 92.8% accuracy, above the 81% target, and finds zero subscribers. Precision, recall, and F1 are all 0. Accuracy can't tell this model apart from a useful one, so the models below are compared on precision and recall for "yes". Accuracy is still reported.

## Preprocessing

In [6]:
categorical_cols = X_train.select_dtypes(include="str").columns.tolist()
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical_cols),
    ("numeric", StandardScaler(), numeric_cols),
])

categorical_cols, numeric_cols

(['job',
  'marital',
  'education',
  'default',
  'housing',
  'loan',
  'contact',
  'month'],
 ['age', 'balance', 'day', 'duration', 'campaign'])

## Logistic regression

In [ ]:
log_reg = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
log_reg_scores = cross_validate(log_reg, X_train, y_train, cv=cv, scoring=scoring)

pd.DataFrame(log_reg_scores).filter(like="test_").mean().round(3)

test_accuracy     0.865
test_precision    0.334
test_recall       0.860
test_f1           0.481
dtype: float64